<a href="https://colab.research.google.com/github/Prakat-star/FDS/blob/main/lab6%267_bct024.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Example dataset
data = {'Color': ['Red', 'Blue', 'Green', 'Blue', 'Red']}
df = pd.DataFrame(data)

In [ ]:
encoded_df = pd.get_dummies(df, columns=['Color'], drop_first=False)
print(encoded_df)

   Color_Blue  Color_Green  Color_Red
0       False        False       True
1        True        False      False
2       False         True      False
3        True        False      False
4       False        False       True


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Example
label_encoder = LabelEncoder()
df['Color_Label'] = label_encoder.fit_transform(df['Color'])
print(df)

   Color  Color_Label
0    Red            2
1   Blue            0
2  Green            1
3   Blue            0
4    Red            2


In [ ]:
frequency_map = df['Color'].value_counts().to_dict()
df['Color_Frequency'] = df['Color'].map(frequency_map)

In [ ]:
print(df)


   Color  Color_Label  Color_Frequency
0    Red            2                2
1   Blue            0                2
2  Green            1                1
3   Blue            0                2
4    Red            2                2


In [ ]:
# Add the missing columns 'Feature1' and 'Feature2' to the DataFrame
# Example values are used, you should replace them with your actual data
import numpy as np
df['Feature1'] = np.random.rand(len(df))
df['Feature2'] = np.random.rand(len(df))

# Now you can proceed with PolynomialFeatures
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, interaction_only=False, include_bias=False)
poly_features = poly.fit_transform(df[['Feature1', 'Feature2']])
poly_features

array([[0.43683586, 0.61007211, 0.19082557, 0.26650138, 0.37218798],
       [0.02767323, 0.66141321, 0.00076581, 0.01830344, 0.43746744],
       [0.03932851, 0.1873215 , 0.00154673, 0.00736708, 0.03508935],
       [0.57393258, 0.46301773, 0.32939861, 0.26574096, 0.21438542],
       [0.2460816 , 0.51422102, 0.06055615, 0.12654033, 0.26442325]])

In [ ]:
# Add a 'Value' column to the DataFrame with example data
# Replace this with your actual data if you have it
df['Value'] = np.random.rand(len(df))

# Now calculate the rolling mean and standard deviation
df['Rolling_Mean'] = df['Value'].rolling(window=3).mean()
df['Rolling_Std'] = df['Value'].rolling(window=3).std()

In [ ]:
# Convert 'Color', 'Color_Label' and 'Color_Frequency' to numeric if possible.
# If not possible to convert, drop them before calculating correlation.
# df['Target'] = np.random.rand(len(df))

# Option 1: Drop non-numeric columns
numerical_df = df.select_dtypes(include=np.number)
correlation = numerical_df.corr()
print(correlation['Feature1'])

# Option 2: Convert 'Color_Label' and 'Color_Frequency' to numeric
# (assuming they represent ordinal or frequency data) and drop 'Color'
for col in ['Color_Label', 'Color_Frequency']:
  df[col] = pd.to_numeric(df[col], errors='coerce') # Handle potential errors
numerical_df = df.drop(columns=['Color']) # Drop the original 'Color' column
correlation = numerical_df.corr()
print(correlation['Feature1'])

Color_Label        0.084302
Color_Frequency    0.522645
Feature1           1.000000
Feature2           0.221316
Value             -0.121440
Rolling_Mean      -0.938550
Rolling_Std        0.251122
Name: Feature1, dtype: float64
Color_Label        0.084302
Color_Frequency    0.522645
Feature1           1.000000
Feature2           0.221316
Value             -0.121440
Rolling_Mean      -0.938550
Rolling_Std        0.251122
Name: Feature1, dtype: float64


In [ ]:
from sklearn.feature_selection import f_regression, SelectKBest # Import f_regression
import pandas as pd
import numpy as np

# Assuming 'df' is your DataFrame and 'Target' is the target column
# If you don't have a 'Target' column, replace it with your target variable
df['Target'] = np.random.rand(len(df))
# Replace 'Target' with your target column if it exists
# and make sure 'df' contains all necessary data

# Define X and y
X = df.drop(columns=['Target', 'Color']) # Features (excluding the target)
y = df['Target'] # Target variable

# Impute or drop NaN values in X before applying SelectKBest
# Option 1: Drop rows with NaN values
X = X.dropna()
y = y[X.index] # Ensure y is aligned with X after dropping rows

# Option 2: Impute NaN values with the mean of each column
# from sklearn.impute import SimpleImputer
# imputer = SimpleImputer(strategy='mean')
# X = imputer.fit_transform(X)

# Now apply SelectKBest with f_regression for continuous target
X_new = SelectKBest(f_regression, k=5).fit_transform(X, y) # Use f_regression
print(X_new)

[[1.         0.1873215  0.8345362  0.60399962 0.34729819]
 [2.         0.46301773 0.24432076 0.42780428 0.35280087]
 [2.         0.51422102 0.36535304 0.48140334 0.31175201]]


In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression # Import LinearRegression
# ... (Your existing code) ...
model = LinearRegression() # Use LinearRegression instead of LogisticRegression
rfe = RFE(model, n_features_to_select=5)
X_rfe = rfe.fit_transform(X, y)
print(rfe.support_)

[ True  True  True  True  True False False]


In [ ]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.01)
lasso.fit(X, y)
print(lasso.coef_)

[-0.08980172 -0.3981828  -0.         -0.          0.          0.
  0.        ]


In [ ]:
from sklearn.ensemble import RandomForestRegressor # Import RandomForestRegressor

model = RandomForestRegressor() # Use RandomForestRegressor for continuous target
model.fit(X, y)
importance = model.feature_importances_

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)
df

,Color,Color_Label,Color_Frequency,Feature1,Feature2,Value,Rolling_Mean,Rolling_Std,Target
0,Red,2,2,0.436836,0.610072,0.772907,NaN,NaN,0.481601
1,Blue,0,2,0.027673,0.661413,0.204556,NaN,NaN,0.547062
2,Green,1,1,0.039329,0.187322,0.834536,0.604000,0.347298,0.955564
3,Blue,0,2,0.573933,0.463018,0.244321,0.427804,0.352801,0.617183
4,Red,2,2,0.246082,0.514221,0.365353,0.481403,0.311752,0.407579


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
train = pd.read_csv('/content/drive/MyDrive/foundation of data science/Practicals/train.csv', index_col = 'PassengerId')
test = pd.read_csv('/content/drive/MyDrive/foundation of data science/Practicals/test.csv')
train.head()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Generate synthetic data for simple linear regression
np.random.seed(42)
X_simple = np.random.rand(100, 1) * 10 # Feature
y_simple = 3 * X_simple + 7 + np.random.randn(100, 1) * 2 # Target with noise

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X_simple, y_simple, test_size=0.2)

# Build and train the simple linear regression model
simple_lr = LinearRegression()
simple_lr.fit(X_train, y_train)

# Predict on test data
y_pred_simple = simple_lr.predict(X_test)

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(X_test, y_test, color='blue', label='Actual Data')
plt.plot(X_test, y_pred_simple, color='red', label='Regression Line')
plt.title('Simple Linear Regression')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.show()

In [ ]:
residuals_simple = y_test - y_pred_simple
plt.figure(figsize=(10, 6))
plt.scatter(y_pred_simple, residuals_simple, color='purple')
plt.axhline(y=0, color='black', linestyle='--', linewidth=1)
plt.title('Residual Plot - Simple Linear Regression')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.show()

In [ ]:
np.random.seed(42)

X_multi = np.random.rand(100, 3) * 10  # 3 features

y_multi = 4 * X_multi[:, 0] + 3 * X_multi[:, 1] - 2 * X_multi[:, 2] + 5 + np.random.randn(100)

# Split the data
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X_multi, y_multi, test_size=0.2, random_state=42
)

# Build and train the multiple linear regression model
multi_lr = LinearRegression()
multi_lr.fit(X_train_multi, y_train_multi)

# Predict on test data
y_pred_multi = multi_lr.predict(X_test_multi)

In [ ]:
# Choose one feature to plot (e.g., Feature 0)
feature_index = 0  # Index of the feature to visualize
fixed_features = np.mean(X_test_multi, axis=0)  # Fix other features at their mean values

# Generate values for the selected feature
x_values = np.linspace(
    X_test_multi[:, feature_index].min(),
    X_test_multi[:, feature_index].max(),
    100
)

# Create input matrix for prediction
X_plot = np.tile(fixed_features, (100, 1))
X_plot[:, feature_index] = x_values

# Predict using the trained model
y_plot = multi_lr.predict(X_plot)

In [ ]:
# Create data points for predictions by fixing other features
X_plot = np.tile(fixed_features, (x_values.shape[0], 1))  # Copy fixed features
X_plot[:, feature_index] = x_values  # Replace selected feature with varying values

# Predict using the model
y_plot = multi_lr.predict(X_plot)

# Scatter plot of actual data points for the selected feature
plt.figure(figsize=(10, 6))
plt.scatter(
    X_test_multi[:, feature_index],
    y_test_multi,
    color='blue',
    label='Actual Data'
)

# Plot the regression line
plt.plot(
    x_values,
    y_plot,
    color='red',
    label='Regression Line',
    linewidth=2
)

# Add labels and title
plt.title(f'Multiple Linear Regression Line for Feature {feature_index + 1}')
plt.xlabel(f'Feature {feature_index + 1}')
plt.ylabel('Target Variable')
plt.legend()
plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt

# Choose two features to plot (e.g., Features 0 and 1)
feature_indices = [0, 1]  # Indices of the features to visualize
fixed_features = np.mean(X_test_multi, axis=0)  # Fix other features at their mean values

# Generate a grid of values for the selected features
x1_values = np.linspace(
    X_test_multi[:, feature_indices[0]].min(),
    X_test_multi[:, feature_indices[0]].max(),
    50
)

x2_values = np.linspace(
    X_test_multi[:, feature_indices[1]].min(),
    X_test_multi[:, feature_indices[1]].max(),
    50
)

x1_grid, x2_grid = np.meshgrid(x1_values, x2_values)

# Create data points for predictions by fixing other features
X_plot = np.tile(fixed_features, (x1_grid.size, 1))  # Copy fixed features
X_plot[:, feature_indices[0]] = x1_grid.ravel()
X_plot[:, feature_indices[1]] = x2_grid.ravel()

# Predict using the model
y_plot = multi_lr.predict(X_plot).reshape(x1_grid.shape)

In [ ]:
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot the regression surface
ax.plot_surface(x1_grid, x2_grid, y_plot, cmap='viridis', alpha=0.7)

# Scatter the actual data points
ax.scatter(
    X_test_multi[:, feature_indices[0]],
    X_test_multi[:, feature_indices[1]],
    y_test_multi,
    color='red',
    label='Actual Data'
)

# Add labels and title
ax.set_title('Multiple Linear Regression Hyperplane')
ax.set_xlabel(f'Feature {feature_indices[0] + 1}')
ax.set_ylabel(f'Feature {feature_indices[1] + 1}')
ax.set_zlabel('Target Variable')

ax.legend()
plt.show()

In [ ]:
residuals_multi = y_test_multi - y_pred_multi
plt.figure(figsize=(10, 6))
plt.scatter(y_pred_multi, residuals_multi, color='green')
plt.axhline(y=0, color='black', linestyle='--', linewidth=1)
plt.title('Residual Plot - Multiple Linear Regression')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.show()

In [ ]:
print("Simple Linear Regression Metrics:")
print(f"Mean Squared Error: {mean_squared_error(y_test, y_pred_simple):.2f}")
print(f"R2 Score: {r2_score(y_test, y_pred_simple):.2f}\n")
print("Multiple Linear Regression Metrics:")
print(f"Mean Squared Error: {mean_squared_error(y_test_multi, y_pred_multi):.2f}")
print(f"R2 Score: {r2_score(y_test_multi, y_pred_multi):.2f}")

In [ ]:
import pandas as pd

In [ ]:
df_raw = pd.read_csv('/content/Student_Performance.csv')

In [ ]:
df_raw.head()

In [ ]:
df_raw.shape


In [ ]:
df_raw.info()


In [ ]:
df_raw.describe()


In [ ]:
df_raw.duplicated().sum()


In [ ]:
df_raw.drop_duplicates(inplace=True)


In [ ]:
df_raw['Extracurricular Activities'].value_counts()


In [ ]:
df_raw['Ext Act'] = df_raw['Extracurricular Activities'].replace({'Yes':1, 'No':0})

df_raw.head()

In [ ]:
df_raw.describe()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
g0 =sns.boxplot(data=df_raw[['Hours Studied','Sleep Hours','Sample Question Papers Practiced']] )
g0.set_title("Check for Outliers")
plt.xticks(rotation=45, ha='right')

In [ ]:
sns.pairplot(df_raw)

In [ ]:
sns.boxplot(data=df_raw, x='Hours Studied', y='Previous Scores', hue='Ext Act')
plt.title('Hours Studied vs Previous Scores')
plt.legend(loc='lower right')


In [ ]:
sns.boxplot(data=df_raw, x='Sleep Hours', y='Previous Scores', hue='Ext Act')
plt.title('Sleep Hours vs Previous Scores')
plt.legend(loc='lower right')

In [ ]:
sns.boxplot(data=df_raw, x='Sample Question Papers Practiced', y='Previous Scores', hue='Ext Act')
plt.title('Sample Question Papers Practiced vs Previous Scores')
plt.legend(loc='lower right')


In [ ]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

In [ ]:
df_mls = df_raw.copy()
df_mls = df_mls.drop(columns=['Ext Act'])

df_mls = df_mls.rename(columns={
    "Hours Studied": "Hours_Studied",
    "Previous Scores": "Previous_Scores",
    "Sleep Hours": "Sleep_Hours",
    "Sample Question Papers Practiced": "Sample_Question_Papers_Practiced",
    "Performance Index": "Performance_Index"
})

df_mls.head()

In [ ]:
# Import train-test-split function from sci-kit learn
from sklearn.model_selection import train_test_split

df_mls_y = df_mls['Performance_Index']
df_mls_X = df_mls.drop(columns=['Performance_Index'])

df_mls_X.shape , df_mls_y.shape

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df_mls_X,
    df_mls_y,
    test_size=0.3,
    random_state=42
)

In [ ]:
X_multi = np.random.rand(100, 3) * 10
y_multi = 4 * X_multi[:, 0] + 3 * X_multi[:, 1] - 2 * X_multi[:, 2] + 5 + np.random.randn(100) * 2

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_multi, y_multi, test_size=0.2, random_state=42)

multi_lr = LinearRegression()
multi_lr.fit(X_train_m, y_train_m)
y_pred_m = multi_lr.predict(X_test_m)

residuals_multi = y_test_m - y_pred_m
plt.figure(figsize=(8, 5))
plt.scatter(y_pred_m, residuals_multi, color='green')
plt.axhline(y=0, color='black', linestyle='--', linewidth=1)
plt.title('Residual Plot - Multiple Linear Regression')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.show()

print("Multiple Linear Regression Metrics:")
print(f"Mean Squared Error: {mean_squared_error(y_test_m, y_pred_m):.2f}")
print(f"R2 Score: {r2_score(y_test_m, y_pred_m):.2f}\n")

In [ ]:
n_students = 1000
df_raw = pd.DataFrame({
    'Hours_Studied': np.random.randint(1, 10, n_students),
    'Previous_Scores': np.random.randint(40, 100, n_students),
    'Extracurricular_Activities': np.random.choice(['Yes', 'No'], n_students),
    'Sleep_Hours': np.random.randint(4, 10, n_students),
    'Sample_Question_Papers_Practiced': np.random.randint(0, 10, n_students)
})

In [ ]:
ext_act_multiplier = df_raw['Extracurricular_Activities'].map({'Yes': 2, 'No': 0})
df_raw['Performance_Index'] = (
    df_raw['Hours_Studied'] * 2.8 +
    df_raw['Previous_Scores'] * 1.0 +
    df_raw['Sleep_Hours'] * 0.5 +
    df_raw['Sample_Question_Papers_Practiced'] * 0.2 +
    ext_act_multiplier - 30 + np.random.randn(n_students) * 2
)

In [ ]:
df_mls_y = df_raw['Performance_Index']
df_mls_X = df_raw.drop(columns=['Performance_Index'])
X_tr, X_te, y_tr, y_te = train_test_split(df_mls_X, df_mls_y, test_size=0.3, random_state=42)

In [ ]:
mls_data = pd.concat([X_tr, y_tr], axis=1)
mls_formula = 'Performance_Index ~ Hours_Studied + Previous_Scores + C(Extracurricular_Activities) + Sleep_Hours + Sample_Question_Papers_Practiced'

In [ ]:
mls_formula = 'Performance_Index ~ Hours_Studied + Previous_Scores + C(Extracurricular_Activities) + Sleep_Hours + Sample_Question_Papers_Practiced'

In [ ]:
mls_data = pd.concat([X_train, y_train], axis = 1)


In [ ]:
# Fit Model
MLS = ols(formula=mls_formula, data=mls_data)
model = MLS.fit()
print(model.summary())

In [ ]:
new_data = pd.DataFrame({
    'Hours_Studied': [5],
    'Previous_Scores': [85],
    'Extracurricular_Activities': ['Yes'],
    'Sleep_Hours': [8],
    'Sample_Question_Papers_Practiced': [5]
})
predicted_values = model.predict(new_data)
print("\nPredicted Performance Index for New Data:\n", predicted_values)

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(x=model.fittedvalues, y=model.resid)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Fitted Values v. Residuals")
plt.show()

In [ ]:
# Q-Q Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.histplot(model.resid, ax=axes[0], kde=True)
axes[0].set_title("Histogram of Residuals")
sm.qqplot(model.resid, line='s', ax=axes[1])
axes[1].set_title("Normal QQ Plot")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import statsmodels.api as sm
from statsmodels.formula.api import ols


In [ ]:
np.random.seed(42)
X_simple = np.random.rand(100, 1) * 10
y_simple = 3 * X_simple + 7 + np.random.randn(100, 1) * 2

X_train, X_test, y_train, y_test = train_test_split(X_simple, y_simple, test_size=0.2, random_state=42)

simple_lr = LinearRegression()
simple_lr.fit(X_train, y_train)
y_pred_simple = simple_lr.predict(X_test)

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(X_test, y_test, color='blue', label='Actual Data')
plt.plot(X_test, y_pred_simple, color='red', label='Regression Line')
plt.title('Simple Linear Regression')
plt.legend()
plt.show()

In [ ]:
residuals_simple = y_test - y_pred_simple
plt.figure(figsize=(10, 6))
plt.scatter(y_pred_simple, residuals_simple, color='purple')
plt.axhline(y=0, color='black', linestyle='--', linewidth=1)
plt.title('Residual Plot - Simple Linear Regression')
plt.show()

In [ ]:
X_multi = np.random.rand(100, 3) * 10
y_multi = 4 * X_multi[:, 0] + 3 * X_multi[:, 1] - 2 * X_multi[:, 2] + 5 + np.random.randn(100) * 2

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_multi, y_multi, test_size=0.2, random_state=42)

multi_lr = LinearRegression()
multi_lr.fit(X_train_m, y_train_m)
y_pred_multi = multi_lr.predict(X_test_m)

print("Simple LR R2 Score:", r2_score(y_test, y_pred_simple))
print("Multiple LR R2 Score:", r2_score(y_test_m, y_pred_multi))

In [ ]:
n_samples = 10000
df_raw = pd.DataFrame({
    'Hours Studied': np.random.randint(1, 10, n_samples),
    'Previous Scores': np.random.randint(40, 100, n_samples),
    'Extracurricular Activities': np.random.choice(['Yes', 'No'], n_samples),
    'Sleep Hours': np.random.randint(4, 10, n_samples),
    'Sample Question Papers Practiced': np.random.randint(0, 10, n_samples)
})
ext_act_mult = df_raw['Extracurricular Activities'].map({'Yes': 1, 'No': 0})
df_raw['Performance Index'] = (
    df_raw['Hours Studied'] * 2.8 + df_raw['Previous Scores'] * 1.0 +
    df_raw['Sleep Hours'] * 0.5 + df_raw['Sample Question Papers Practiced'] * 0.2 +
    ext_act_mult - 34 + np.random.randn(n_samples) * 2
)

In [ ]:
# Preprocessing
df_raw.drop_duplicates(inplace=True)
df_mls = df_raw.rename(columns={
    "Hours Studied": "Hours_Studied", "Previous Scores": "Previous_Scores",
    "Extracurricular Activities": "Extracurricular_Activities", "Sleep Hours": "Sleep_Hours",
    "Sample Question Papers Practiced": "Sample_Question_Papers_Practiced", "Performance Index": "Performance_Index"
})

In [ ]:
# Train Test Split
df_mls_y = df_mls['Performance_Index']
df_mls_X = df_mls.drop(columns=['Performance_Index'])
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(df_mls_X, df_mls_y, test_size=0.3, random_state=42)

In [ ]:
mls_data = pd.concat([X_train_s, y_train_s], axis=1)
mls_formula = 'Performance_Index ~ Hours_Studied + Previous_Scores + C(Extracurricular_Activities) + Sleep_Hours + Sample_Question_Papers_Practiced'
MLS = ols(formula=mls_formula, data=mls_data)
model = MLS.fit()
print("\n", model.summary())

In [ ]:
# Prediction on new data
new_data = pd.DataFrame({
    'Hours_Studied': [5],
    'Previous_Scores': [85],
    'Extracurricular_Activities': ['Yes'],
    'Sleep_Hours': [8],
    'Sample_Question_Papers_Practiced': [5]
})
print("\nPredicted Value for new data:\n", model.predict(new_data))

In [ ]:
# Residual Diagnostics
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.histplot(model.resid, ax=axes[0], kde=True)
axes[0].set_title("Histogram of Residuals")
sm.qqplot(model.resid, line='s', ax=axes[1])
axes[1].set_title("Normal QQ Plot")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(x=model.fittedvalues, y=model.resid)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Fitted Values v. Residuals")
plt.show()